# 02. U-Net trained from scratch

Plain 2D U-Net (base 32 filters, depth 4, no dropout) on the three lidar bands, loss BCE + Dice with equal weights. Config: `configs/unet_scratch.yaml`.

## Setup

On Colab this mounts Google Drive, clones the repo and installs it. Locally it assumes
`pip install -e .` has been run in the repo and the tiles are in `data/chactun/`.

In [ ]:
REPO_URL = "https://github.com/skumagai3/maya-lidar.git"
DRIVE_DIR = "/content/drive/MyDrive/maya-lidar"           # Colab: runs are saved here
DATA_ZIP = f"{DRIVE_DIR}/data/22202395.zip"               # Colab: Figshare bundle, or a zip of the tile files

import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/maya-lidar")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))  # an editable install is not importable until restart
else:
    DATA_ZIP = None

import numpy as np
import matplotlib.pyplot as plt
from mayalidar import config as C, data as D, viz as V

paths = C.project_paths(DRIVE_DIR)
D.ensure_extracted(DATA_ZIP, paths.data_dir)
tiles = D.index_tiles(paths.data_dir)
print(f"{len(tiles)} tiles in {paths.data_dir}; runs go to {paths.runs_dir}")

## Config and split

In [ ]:
import keras
from mayalidar import train as T, evaluate as E

cfg = C.load_config("unet_scratch")
train, val, test = D.official_split(tiles, cfg["data"]["val_frac"], cfg["data"]["split_seed"])
print(f"train {len(train)}, validation {len(val)}, test {len(test)}")
print(cfg)

## Smoke test

A few tiles and epochs through the full training and scoring path, so problems show up
before the long run. Outputs go to `runs/<experiment>/<timestamp>-smoke/`.

In [ ]:
T.gpu_report(require=True)
smoke_dir = C.new_run_dir(paths, cfg["experiment"], "smoke")
net, stats = T.run(cfg, train, val, smoke_dir, smoke=True)
ds = D.make_dataset(val[: cfg["smoke"]["n_val"]], stats, T.MASK_PRESENT_VALUE, cfg["train"]["batch_size"])
Y, S = E.predict(net, ds)
print("labels", Y.shape, "sigmoid outputs", S.shape, "range", float(S.min()), float(S.max()))
assert np.isfinite(S.astype(np.float32)).all()
print(f"peak GPU memory: {T.peak_gpu_memory_gb():.1f} GB at batch size {cfg['train']['batch_size']}")
print("smoke test passed:", smoke_dir)

## Full training run

Saves `config.yaml`, `split.json`, `normalization.json`, `log.csv` and `best.keras`
(lowest validation loss) to `runs/<experiment>/<timestamp>/`. If the Colab session
drops, set `RESUME` to that folder's name and rerun from the top: training continues
from the last finished epoch.

In [ ]:
RESUME = None  # e.g. "20261113-091500" to continue an interrupted run
run_dir = paths.runs_dir / cfg["experiment"] / RESUME if RESUME else C.new_run_dir(paths, cfg["experiment"])
net, stats = T.run(cfg, train, val, run_dir)
fig = V.plot_history([run_dir / "log.csv", run_dir / "log_finetune.csv"],
                     f"{cfg['experiment']}: loss and validation IoU per epoch")
V.save(fig, run_dir, "training_curves")
plt.show()

## Validation: per-class thresholds

The checkpoint with the lowest validation loss is reloaded. Its sigmoid outputs on
validation are saved to `val_outputs.npz` (about 120 MB) for the viewer notebook. A pixel counts as class c
when the sigmoid output exceeds a threshold; the threshold for each class is chosen
here to maximize validation IoU, then held fixed for the test set.

In [ ]:
best = keras.models.load_model(run_dir / "best.keras", compile=False)
bs = cfg["train"]["batch_size"]
Yv, Sv = E.predict(best, D.make_dataset(val, stats, T.MASK_PRESENT_VALUE, bs))
E.save_outputs(run_dir / "val_outputs.npz", [t.tile_id for t in val], Sv)  # read by 04_viewer
sweep = E.threshold_sweep(Yv, Sv, cfg["eval"]["thresholds"])
chosen = E.best_thresholds(sweep)
val_scores = E.scores(Yv, Sv, chosen)
fig = V.plot_threshold_sweep(sweep, chosen, f"{cfg['experiment']}: validation IoU vs. threshold (star = chosen)")
V.save(fig, run_dir, "threshold_sweep_val")
plt.show()
print(val_scores)

## Test set (tiles 1765-2093)

Scored once, with the thresholds chosen on validation. Nothing about the model or the
thresholds is tuned on these tiles. IoU at a fixed threshold of 0.5 is also saved, for
comparison with runs that do not tune thresholds.

In [ ]:
Yt, St = E.predict(best, D.make_dataset(test, stats, T.MASK_PRESENT_VALUE, bs))
test_scores = E.scores(Yt, St, chosen)
results = {"validation": val_scores, "test": test_scores,
           "validation_at_0.5": E.scores(Yv, Sv, 0.5), "test_at_0.5": E.scores(Yt, St, 0.5),
           "threshold_sweep_val": sweep}
D.save_json(results, run_dir / "scores.json")
fig = V.plot_scores({"validation": val_scores, "test": test_scores},
                    f"{cfg['experiment']}: IoU per class at validation-chosen thresholds")
V.save(fig, run_dir, "iou_val_test")
plt.show()
print("test:", test_scores)

## Diagnostics on validation

Pixel precision-recall per class, then an object-level view. Objects are connected
regions of the labels or the prediction. A labeled object counts as detected when at
least half its pixels are predicted, and a predicted object counts as false when none
of its pixels are labeled. Results are saved to `diagnostics.json`.

In [ ]:
pr = E.pr_curves(Yv, Sv)
obj = E.object_size_stats(Yv, Sv, chosen)
D.save_json({"pr": pr, "object_size": obj}, run_dir / "diagnostics.json")
fig = V.plot_pr_curves(pr, chosen, f"{cfg['experiment']}: validation pixel precision vs. recall (dots = chosen thresholds)")
V.save(fig, run_dir, "pr_curves_val")
plt.show()
fig = V.plot_object_size(obj, f"{cfg['experiment']}: validation objects by area, detected (left) and false (right)")
V.save(fig, run_dir, "object_size_val")
plt.show()

## Galleries of validation tiles

Worst tiles by mean IoU over the classes present, then for each class the tiles with
the most missed pixels and with the most false-positive pixels.

In [ ]:
def show_gallery(order, title, name, cls=None, n=12):
    idx = [k for k in order[:n]]
    xs = [D.read_tif(val[k].lidar) for k in idx]
    titles = [f"tile {val[k].tile_id}" for k in idx]
    fig = V.gallery(xs, Yv[idx], Sv[idx].astype(np.float32), chosen, titles, title, cls=cls)
    V.save(fig, run_dir, name)
    plt.show()

per_tile = E.per_tile_mean_iou(Yv, Sv, chosen)
order = np.argsort(np.nan_to_num(per_tile, nan=1.0))
show_gallery(order, f"{cfg['experiment']}: 12 worst validation tiles by mean IoU, labels (solid) and predictions (dashed)",
             "gallery_worst_val")
for c in D.CLASSES:
    err = E.per_tile_errors(Yv, Sv, chosen, c)
    show_gallery(np.argsort(-err["fn"]), f"{cfg['experiment']}: validation tiles with the most missed {c} pixels",
                 f"gallery_missed_{c}", cls=c, n=8)
    show_gallery(np.argsort(-err["fp"]), f"{cfg['experiment']}: validation tiles with the most false-positive {c} pixels",
                 f"gallery_false_{c}", cls=c, n=8)

## The three worst validation tiles in detail

In [ ]:
for k in order[:3]:
    t = val[k]
    fig = V.show_tile(D.read_tif(t.lidar), Yv[k], f"Validation tile {t.tile_id}: inputs, sigmoid outputs and errors",
                      s=Sv[k].astype(np.float32), thresholds=chosen)
    V.save(fig, run_dir, f"tile_{t.tile_id}_val")
    plt.show()
print("all outputs in", run_dir)